In [2]:
import pandas as pd
import numpy as np

In [3]:
Converted_Fare=pd.read_excel("../Data/BUS/Converted_Fare.xlsx",engine="openpyxl")

In [4]:
# 选择上下车站都在同一个区的路线
Converted_Fare=Converted_Fare[Converted_Fare['ON_DISTRICT']==Converted_Fare['OFF_DISTRICT']]
# 计算每个组的OFF_SEQ - ON_SEQ值
Converted_Fare['SEQ_DIFF'] = np.abs(Converted_Fare['OFF_SEQ'] - Converted_Fare['ON_SEQ'])

# 找到每个组中SEQ_DIFF最大值所对应的行的索引
idx = Converted_Fare.groupby(['ROUTE_ID', 'ROUTE_SEQ', 'PRICE','ON_DISTRICT'])['SEQ_DIFF'].idxmax()

# 使用这些索引来获取每个组中SEQ_DIFF最大值所对应的数据
result = Converted_Fare.loc[idx].drop(columns=['SEQ_DIFF'])


In [5]:
Selected=result.copy()

In [6]:
Selected=Selected.drop(columns=['LAST_UPDATE_DATE','STOP_SEQ_x','STOP_SEQ_y'])
Selected=Selected.rename(columns={'ON_SEQ_STOP_Y':'ON_STOP_Y','OFF_SEQ_STOP_Y':'OFF_STOP_Y','ON_SEQ_STOP_X':'ON_STOP_X','OFF_SEQ_STOP_X':'OFF_STOP_X'})
Selected

,ROUTE_ID,ROUTE_SEQ,ON_SEQ,OFF_SEQ,PRICE,ROUTE_NAMES,COMPANY_CODE,STOP_NAMES_x,ON_SEQ_STOP_ID,STOP_NAMES_y,OFF_SEQ_STOP_ID,ON_STOP_X,ON_STOP_Y,OFF_STOP_X,OFF_STOP_Y,ON_DISTRICT,OFF_DISTRICT
271,1001,1,17,25,5.8,1,KMB,旺角奶路臣街,7036,尖沙咀码头,4025,835531,819900,835463,817244,油尖旺,油尖旺
152,1001,1,8,14,6.7,1,KMB,九龙寨城公园,4008,拔萃男书院,4014,837429,821474,836023,820663,九龍城,九龍城
254,1001,1,15,25,6.7,1,KMB,协和小学,4015,尖沙咀码头,4025,835862,820632,835463,817244,油尖旺,油尖旺
5,1001,1,1,7,6.7,1,KMB,竹园邨总站,4001,盈东楼,8090,837872,822918,837727,821666,黃大仙,黃大仙
525,1001,2,13,17,6.3,1,KMB,明爱太子宿舍,4077,东宝庭道,4039,836322,820884,837258,821296,九龍城,九龍城
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755777,1000575,1,1,18,5.1,272A,KMB,大学站,6236,大学站,6236,839753,830365,839753,830365,沙田,沙田
729030,1000587,1,16,17,5.4,270B,KMB,上水转车站 - 上水站,6626,上水转车站 - 上水总站,6625,831178,840270,831219,840332,北區,北區
729025,1000587,1,14,15,16.9,270B,KMB,青沙公路转车站,13378,青沙公路转车站,13185,834816,824550,834844,824598,沙田,沙田
728206,1000587,1,1,6,16.9,270B,KMB,奥运站巴士总站,9048,太子站,9077,834607,819986,835345,820751,油尖旺,油尖旺


In [7]:
Selected['DISTANCE'] = np.sqrt((Selected['ON_STOP_X'] - Selected['OFF_STOP_X'])**2 + 
                                  (Selected['ON_STOP_Y'] - Selected['OFF_STOP_Y'])**2)

In [8]:
#筛选出合理的范围：
#假设区之间人们只要PRICE<10都接受
Selected=Selected[Selected['PRICE']<10]

#筛选Distance不为0的路线
Selected=Selected[Selected['DISTANCE']!=0]
# 计算每个区域的总距离
district_total_distance = Selected.groupby("ON_DISTRICT")["DISTANCE"].transform("sum")

# 计算成本指数
Selected["COST_INDEX"] = (Selected["PRICE"] * Selected["DISTANCE"]) / district_total_distance

In [9]:
Selected

,ROUTE_ID,ROUTE_SEQ,ON_SEQ,OFF_SEQ,PRICE,ROUTE_NAMES,COMPANY_CODE,STOP_NAMES_x,ON_SEQ_STOP_ID,STOP_NAMES_y,OFF_SEQ_STOP_ID,ON_STOP_X,ON_STOP_Y,OFF_STOP_X,OFF_STOP_Y,ON_DISTRICT,OFF_DISTRICT,DISTANCE,COST_INDEX
271,1001,1,17,25,5.8,1,KMB,旺角奶路臣街,7036,尖沙咀码头,4025,835531,819900,835463,817244,油尖旺,油尖旺,2656.870339,0.030104
152,1001,1,8,14,6.7,1,KMB,九龙寨城公园,4008,拔萃男书院,4014,837429,821474,836023,820663,九龍城,九龍城,1623.131849,0.022507
254,1001,1,15,25,6.7,1,KMB,协和小学,4015,尖沙咀码头,4025,835862,820632,835463,817244,油尖旺,油尖旺,3411.413930,0.044651
5,1001,1,1,7,6.7,1,KMB,竹园邨总站,4001,盈东楼,8090,837872,822918,837727,821666,黃大仙,黃大仙,1260.368597,0.028666
525,1001,2,13,17,6.3,1,KMB,明爱太子宿舍,4077,东宝庭道,4039,836322,820884,837258,821296,九龍城,九龍城,1022.663190,0.013334
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
728851,1000574,2,30,40,4.8,E36C,LWB,凤池村,9972,元朗(德业街),10000041,820054,834112,821245,834648,元朗,元朗,1306.053981,0.014569
727925,1000574,2,9,29,6.8,E36C,LWB,蝴蝶湾,12871,五柳路,13005,813853,826289,815971,830709,屯門,屯門,4901.257390,0.049780
728191,1000575,1,13,18,4.6,272A,KMB,科研路,13089,大学站,6236,839559,831876,839753,830365,沙田,沙田,1523.403098,0.007967
755819,1000575,1,7,12,5.1,272A,KMB,云滙,13002,创新路,10000017,838993,832234,839162,832142,大埔,大埔,192.418814,0.002973


In [10]:
district_mean_cost = Selected.groupby('ON_DISTRICT')['COST_INDEX'].mean().reset_index()
district_mean_cost['COST_INDEX']=district_mean_cost['COST_INDEX']*100
# 重命名列以便更清晰地表示
district_mean_cost.rename(columns={'COST_INDEX': 'MEAN_COST_INDEX(%)'}, inplace=True)

# 显示结果
print(district_mean_cost)

   ON_DISTRICT  MEAN_COST_INDEX(%)
0          中西區            1.841896
1          九龍城            3.075893
2           元朗            4.273722
3           北區            5.492941
4           南區            1.972500
5           大埔            5.070724
6           屯門            3.781400
7           東區            2.125092
8           沙田            2.588576
9          油尖旺            2.164484
10         深水埗            2.953228
11          灣仔            1.782159
12          荃灣            3.696629
13          葵青            2.247986
14          西貢            5.558570
15          觀塘            2.010924
16          離島            2.126161
17         黃大仙            3.190044


In [40]:
result=district_mean_cost

In [41]:
Selected.to_excel("../Data/BUS/SelectedRoute-2.0.xlsx",engine='openpyxl')
result.to_excel("../Data/BUS/Cost_index_18.xlsx",engine="openpyxl")